# Options from Zero · Episode 10: The Black-Scholes formula, decoded

Companion notebook for the video. Every number that appears on screen or in the narration is produced by running this notebook and read from `build/outputs.json`.

> **The running trade.** An illustrative share priced at \$100 that pays no dividends, a one-year call option on it with a strike of \$105, 20% volatility and a 3.60% interest rate (continuously compounded). All made up for teaching; it is not market data.
>
> Educational material only, not financial advice. Options are risky: a seller can lose far more than the premium received.

1. Setup · 2. Last episode's question · 3. The ingredients · 4. N(d2): what you pay · 5. N(d1): what you get · 6. The price · 7. The tree agrees · 8. What isn't in the formula · 9. History · 10. Part 1 checkpoint · 11. Check against QuantLib · 12. Export

**Running in Google Colab?** Run the next cells first: they install QuantLib (version 1.43, the one used in the video) and write the data files this notebook reads. Then run the rest of the notebook in order.

In [ ]:
# Colab doesn't include QuantLib. Install it (about 30 seconds).
# The video used QuantLib 1.43; drop '==1.43' for the latest.
!pip install QuantLib==1.43

In [ ]:
# Parameters (papermill overrides these)
output_json = "build/outputs.json"

## 1. Setup

In [ ]:
import json
import math
import datetime as dt
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import QuantLib as ql

out = {"meta": {
    "series": "Options from Zero",
    "episode": 10,
    "quantlib_version": ql.__version__,
    "quotes_label": "illustrative (made up for teaching)",
    "generated_at": dt.datetime.now().isoformat(timespec="seconds"),
}}
S, K, T, r, vol = 100.0, 105.0, 1.0, 0.036, 0.20   # the running trade

The option formulas used throughout the series (`episodes/_shared/options.py`):

In [ ]:
# Option formulas for Options from Zero, written out by hand so every step can be read. Copied verbatim into
# each notebook by make_notebook.py, so every notebook runs on its own; the notebooks check these against
# QuantLib. Needs: math, numpy as np.
#   Running trade (Part 1): an illustrative share at $100, no dividends, 20% volatility, a one-year call struck
#   at $105, interest 3.60% a year continuously compounded. Made up for teaching.

def norm_cdf(x):
    """N(x): the chance a standard normal variable is below x."""
    return 0.5 * (1 + math.erf(x / math.sqrt(2)))

def call_payoff(s, k):
    return np.maximum(np.asarray(s, dtype=float) - k, 0.0)

def put_payoff(s, k):
    return np.maximum(k - np.asarray(s, dtype=float), 0.0)

def black_scholes(s, k, t, r, vol, call=True):
    """Black-Scholes price of a European option on a share with no dividends (r continuously compounded)."""
    d1 = (math.log(s / k) + (r + vol ** 2 / 2) * t) / (vol * math.sqrt(t))
    d2 = d1 - vol * math.sqrt(t)
    if call:
        return s * norm_cdf(d1) - k * math.exp(-r * t) * norm_cdf(d2)
    return k * math.exp(-r * t) * norm_cdf(-d2) - s * norm_cdf(-d1)

def one_step(s, up, down, k, t, r, call=True):
    """One-step tree: replicate the option with delta shares and cash. Returns the pieces and the price."""
    pay = call_payoff if call else put_payoff
    v_up, v_down = float(pay(up, k)), float(pay(down, k))
    delta = (v_up - v_down) / (up - down)
    cash = (v_down - delta * down) * math.exp(-r * t)   # negative: borrowed today
    q = (s * math.exp(r * t) - down) / (up - down)       # risk-neutral probability of the up move
    return {"v_up": v_up, "v_down": v_down, "delta": delta, "cash": cash, "price": delta * s + cash, "q": q}

def binomial_tree(s, k, t, r, vol, steps, call=True, american=False, quantlib_p=False):
    """Cox-Ross-Rubinstein tree: u = exp(vol sqrt(dt)), d = 1/u, worked backward from expiry.
    quantlib_p=True uses QuantLib's up-probability 1/2 + 1/2 (r - vol^2/2) sqrt(dt) / vol instead of the textbook
    (e^{r dt} - d) / (u - d); both give the same price in the limit."""
    dt = t / steps
    u = math.exp(vol * math.sqrt(dt)); d = 1 / u
    p = 0.5 + 0.5 * (r - vol ** 2 / 2) * math.sqrt(dt) / vol if quantlib_p else (math.exp(r * dt) - d) / (u - d)
    pay = call_payoff if call else put_payoff
    prices = s * d ** np.arange(steps, -1, -1) * u ** np.arange(steps + 1)
    values = pay(prices, k)
    for _ in range(steps):
        prices = prices[1:] / u
        values = math.exp(-r * dt) * (p * values[1:] + (1 - p) * values[:-1])
        if american:
            values = np.maximum(values, pay(prices, k))
    return float(values[0])

## 2. Last episode's question

A share with 16% volatility: a typical daily move is about $16\% / \sqrt{252}$, roughly 1%.

In [ ]:
out["answer"] = {"vol_pct": 16, "daily_pct": 0.16 / math.sqrt(252) * 100}
out["answer"]

## 3. The ingredients

$$C = S\,N(d_1) - K e^{-rT} N(d_2), \qquad d_1 = \frac{\ln(S/K) + (r + \sigma^2/2)T}{\sigma\sqrt{T}}, \qquad d_2 = d_1 - \sigma\sqrt{T}$$

Five inputs: share price, strike, time, interest rate, volatility.

In [ ]:
d1 = (math.log(S / K) + (r + vol ** 2 / 2) * T) / (vol * math.sqrt(T))
d2 = d1 - vol * math.sqrt(T)
out["inputs"] = {"spot": S, "strike": K, "years": T, "rate_pct": round(r * 100, 6), "vol_pct": round(vol * 100, 6),
                 "d1": d1, "d2": d2, "d2_abs": abs(d2), "ln_s_k": math.log(S / K),
                 "rows": [{"input": "Share price", "value": f"${S:.0f}"}, {"input": "Strike", "value": f"${K:.0f}"},
                          {"input": "Time to expiry", "value": f"{T:.0f} year"}, {"input": "Interest rate", "value": f"{r * 100:.2f}%"},
                          {"input": "Volatility", "value": f"{vol * 100:.0f}%"}, {"input": "d1", "value": f"{d1:.4f}"},
                          {"input": "d2", "value": f"{d2:.4f}"}]}
out["inputs"]["rows"]

## 4. N(d2): what you pay

$N(d_2)$ is the chance, under the risk-neutral odds of Episode 6, that the call ends in the money and you pay the strike. So the cash leg is the strike, discounted, times that chance. A Monte Carlo check: simulate a million risk-neutral outcomes and count.

In [ ]:
rng = np.random.default_rng(10)
z = rng.standard_normal(1_000_000)
ST = S * np.exp((r - vol ** 2 / 2) * T + vol * math.sqrt(T) * z)
out["cash_leg"] = {"N_d2": norm_cdf(d2), "N_d2_pct": norm_cdf(d2) * 100, "pv_strike": K * math.exp(-r * T),
                   "value": K * math.exp(-r * T) * norm_cdf(d2), "simulated_share_itm": float((ST > K).mean())}
out["cash_leg"]

## 5. N(d1): what you get

$S\,N(d_1)$ is the value today of the share you receive if you exercise. $N(d_1)$ is also the call's **delta**: Episode 5's number of shares that copies the option, now for a tree with infinitely many steps. It is *not* the probability of exercise (that's $N(d_2)$).

In [ ]:
h = 0.01
bump_delta = (black_scholes(S + h, K, T, r, vol) - black_scholes(S - h, K, T, r, vol)) / (2 * h)
out["share_leg"] = {"N_d1": norm_cdf(d1), "value": S * norm_cdf(d1), "bump_delta": bump_delta,
                    "simulated": float(np.mean(np.where(ST > K, ST, 0.0)) * math.exp(-r * T))}
assert abs(bump_delta - norm_cdf(d1)) < 1e-6
out["share_leg"]

## 6. The price

In [ ]:
price = out["share_leg"]["value"] - out["cash_leg"]["value"]
out["price"] = {"price": price, "formula": black_scholes(S, K, T, r, vol), "share_leg": out["share_leg"]["value"],
                "cash_leg": out["cash_leg"]["value"]}
assert abs(price - black_scholes(S, K, T, r, vol)) < 1e-12
out["price"]

## 7. The tree agrees

Episode 7's tree with more and more steps. Odd and even step counts land on slightly different sides, and the wobble shrinks towards the formula.

In [ ]:
ns = list(range(1, 201))
tree_prices = [binomial_tree(S, K, T, r, vol, n) for n in ns]
table = [{"steps": n, "tree": binomial_tree(S, K, T, r, vol, n)} for n in (10, 11, 100, 101, 1000)]
for row in table:
    row["gap"] = row["tree"] - price
out["tree"] = {"steps": ns, "prices": tree_prices, "formula_line": [price] * len(ns), "table": table,
               "t10": table[0]["tree"], "t11": table[1]["tree"], "t100": table[2]["tree"], "t101": table[3]["tree"],
               "t1000": table[4]["tree"], "gap1000_abs": abs(table[4]["gap"]),
               "ticks": [{"x": float(x), "label": str(x)} for x in (1, 50, 100, 150, 200)]}
assert abs(table[4]["gap"]) < 0.001
pd.DataFrame(table)

## 8. What isn't in the formula

The share's expected return isn't an input: it cancelled out in Episode 3, and again here. Volatility is the only input you can't look up on a screen, which is where Parts 2 and 3 begin. The price for a range of volatilities:

In [ ]:
out["vol_sensitivity"] = {"rows": [{"vol_pct": v * 100, "call": black_scholes(S, K, T, r, v)} for v in (0.10, 0.20, 0.30)]}
out["vol_sensitivity"]["at10"] = out["vol_sensitivity"]["rows"][0]["call"]
out["vol_sensitivity"]["at30"] = out["vol_sensitivity"]["rows"][2]["call"]
pd.DataFrame(out["vol_sensitivity"]["rows"])

## 9. History

Typed from the Nobel Foundation's press release of 14 October 1997 and the original papers: Black and Scholes published the formula in 1973 (*Journal of Political Economy*), with Merton's work the same year (*Bell Journal of Economics and Management Science*); Cox, Ross and Rubinstein published the binomial tree in 1979 (*Journal of Financial Economics*); Merton and Scholes received the 1997 prize in economic sciences; Fischer Black died in 1995.

In [ ]:
out["history"] = {"bs_year": 1973, "crr_year": 1979, "nobel_year": 1997, "black_died": 1995}
out["history"]

## 10. Part 1 checkpoint (answers)

1. The one-year 105 call on a one-step tree with \$130 or \$75: price it by copying and by risk-neutral odds.
2. A 100-strike call costs \$9.72 and the put \$6.18: is parity satisfied?
3. For a share expected to grow faster than the risk-free rate, which is bigger: the real-world or the risk-neutral chance of exercise?

In [ ]:
q1 = one_step(S, 130.0, 75.0, K, T, r)
c100, p100 = black_scholes(S, 100.0, T, r, vol, True), black_scholes(S, 100.0, T, r, vol, False)
out["checkpoint"] = {"q1_delta": q1["delta"], "q1_price": q1["price"], "q1_q": q1["q"],
                     "q1_price_by_q": math.exp(-r * T) * q1["q"] * max(130.0 - K, 0),
                     "q2_call": c100, "q2_put": p100, "q2_lhs": c100 - p100, "q2_rhs": 100.0 - 100.0 * math.exp(-r * T),
                     "q3": "real-world: higher drift pushes more outcomes above the strike"}
out["checkpoint"]

## 11. Check against QuantLib

In [ ]:
today = ql.Date(29, 9, 2026)
ql.Settings.instance().evaluationDate = today
dc = ql.Actual365Fixed()
flat = lambda x: ql.YieldTermStructureHandle(ql.FlatForward(today, x, dc, ql.Continuous))
process = ql.BlackScholesMertonProcess(
    ql.QuoteHandle(ql.SimpleQuote(S)), flat(0.0), flat(r),
    ql.BlackVolTermStructureHandle(ql.BlackConstantVol(today, ql.NullCalendar(), vol, dc)))
expiry = today + 365   # exactly one year on ACT/365F
opt = ql.VanillaOption(ql.PlainVanillaPayoff(ql.Option.Call, K), ql.EuropeanExercise(expiry))
opt.setPricingEngine(ql.AnalyticEuropeanEngine(process))
out["quantlib"] = {"price": opt.NPV(), "delta": opt.delta(), "diff": abs(opt.NPV() - price), "delta_diff": abs(opt.delta() - norm_cdf(d1))}
assert out["quantlib"]["diff"] < 1e-10 and out["quantlib"]["delta_diff"] < 1e-10
out["quantlib"]

## 12. Export for the video

In [ ]:
path = Path(output_json)
path.parent.mkdir(parents=True, exist_ok=True)
path.write_text(json.dumps(out, indent=2, default=float))
print("wrote", path.resolve(), f"({path.stat().st_size / 1024:.0f} KB)")